# Analyses de mobilité CityBikes — direct + historique

Ce notebook complète la Silver CityBikes en direct avec les séries horaires issues de l'archive. Il produit des analyses descriptives : activité indirecte, pics journaliers et stations critiques.

CityBikes fournit des stocks observés, pas les trajets. Une variation de stock est donc un proxy d'activité et ne doit pas être présentée comme un nombre de locations.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'config' / 'cities.yml').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
MIN_OBS = 10
SEUIL_CRITIQUE = 0.30
print('Projet :', PROJECT_ROOT)

## 1. Chargement et période couverte

La clé de déduplication est `(network_id, station_id, snapshot_minute_utc)`. Les archives horaires et les relevés directs peuvent être concaténés lorsque leurs colonnes suivent le contrat Silver.

In [ ]:
def read_many(paths):
    return pd.concat((pd.read_parquet(path) for path in paths), ignore_index=True) if paths else pd.DataFrame()

live_paths = sorted((DATA_ROOT / 'silver' / 'mobility').glob('year=*/month=*/mobility_stations_*.parquet'))
archive_paths = sorted((DATA_ROOT / 'citybikes-archive' / 'silver').glob('year=*/citybikes_stations_hourly.parquet'))
live = read_many(live_paths)
archive = read_many(archive_paths)

if not archive.empty:
    archive['observed_at_utc'] = pd.to_datetime(archive.get('hour_utc'), utc=True)
    archive['snapshot_minute_utc'] = archive['observed_at_utc'].dt.floor('min').astype(str)
if not live.empty:
    live['observed_at_utc'] = pd.to_datetime(live['observed_at_utc'], utc=True)

frames = [frame for frame in (live, archive) if not frame.empty]
mobility = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if mobility.empty:
    print('Aucune Silver mobilité trouvée. Lancez la collecte ou citybikes_archive.py.')
else:
    mobility['bikes_available'] = pd.to_numeric(mobility['bikes_available'], errors='coerce')
    mobility['free_slots'] = pd.to_numeric(mobility['free_slots'], errors='coerce')
    mobility = mobility.drop_duplicates(['network_id', 'station_id', 'snapshot_minute_utc'])
    mobility['hour_local'] = mobility['observed_at_utc'].dt.tz_convert('Europe/Paris').dt.hour
    print(f'{len(mobility):,} observations | {mobility.observed_at_utc.min()} → {mobility.observed_at_utc.max()}')
    display(mobility.groupby(['city_id', 'city_name'], as_index=False).agg(
        stations=('station_id', 'nunique'), observations=('station_id', 'count'),
        debut=('observed_at_utc', 'min'), fin=('observed_at_utc', 'max'),
    ))

## 2. Pics journaliers et activité indirecte

Le profil est calculé sur l'heure locale Europe/Paris. L'activité d'une station est la somme des variations absolues du stock de vélos entre deux observations successives.

In [ ]:
if not mobility.empty:
    ordered = mobility.sort_values(['network_id', 'station_id', 'observed_at_utc']).copy()
    ordered['bike_change_abs'] = ordered.groupby(['network_id', 'station_id'])['bikes_available'].diff().abs()
    profile = ordered.groupby(['city_id', 'hour_local'], as_index=False).agg(
        bikes_mean=('bikes_available', 'mean'),
        free_slots_mean=('free_slots', 'mean'),
        activity_proxy=('bike_change_abs', 'sum'),
        observations=('station_id', 'count'),
    )
    display(profile.sort_values(['city_id', 'activity_proxy'], ascending=[True, False]).groupby('city_id').head(5))
    for city_id, city_profile in profile.groupby('city_id'):
        city_profile = city_profile.sort_values('hour_local')
        city_profile.plot(x='hour_local', y='bikes_mean', marker='o', title=f'{city_id} — stock moyen par heure locale', figsize=(9, 3))
        plt.ylabel('Vélos disponibles')
        plt.show()

## 3. Stations critiques sur la période

Une station est critique si elle est vide ou pleine dans au moins 30 % de ses observations valides et possède au moins 10 observations. Le seuil est réglable dans la cellule de paramétrage.

In [ ]:
if not mobility.empty:
    mobility['valid_bikes'] = mobility['bikes_available'].notna()
    mobility['valid_slots'] = mobility['free_slots'].notna()
    critical = mobility.groupby(['city_id', 'city_name', 'network_id', 'station_id', 'station_name'], dropna=False).agg(
        observations=('station_id', 'size'),
        empty_rate=('bikes_available', lambda s: (s.eq(0).sum() / s.notna().sum()) if s.notna().any() else 0),
        full_rate=('free_slots', lambda s: (s.eq(0).sum() / s.notna().sum()) if s.notna().any() else 0),
        bikes_mean=('bikes_available', 'mean'), free_slots_mean=('free_slots', 'mean'),
    ).reset_index()
    critical['critical_rate'] = critical[['empty_rate', 'full_rate']].max(axis=1)
    critical = critical[(critical['observations'] >= MIN_OBS) & (critical['critical_rate'] >= SEUIL_CRITIQUE)]
    display(critical.sort_values(['city_id', 'critical_rate'], ascending=[True, False]).head(30))
else:
    print('Analyse impossible sans données.')

## Limites et interprétation

- Les données mesurent des stocks, pas les trajets.
- Les résultats dépendent de la période effectivement présente dans les fichiers.
- L'archive CityBikes peut contenir des mois manquants ; ils ne sont pas interpolés.
- Pour la vidéo, afficher la période couverte, le nombre d'observations et le seuil des stations critiques.